# JSSP Phase 1 Results Visualization
## Comparing Branch & Bound vs Dynamic Programming

This notebook visualizes and compares the results from:
- Kaggle B&B (Branch & Bound)
- Google Colab DP (Dynamic Programming)

**Analysis includes**:
- Optimality proof rates
- Computation time comparison
- Instance difficulty analysis
- Algorithm performance profiles

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Visualization setup complete")

In [ ]:
# Load results from both algorithms
# Update these paths to your actual result files
bnb_path = "bnb_phase1_results.csv"  # From Kaggle
dp_path = "dp_phase1_results.csv"    # From Google Colab

try:
    df_bnb = pd.read_csv(bnb_path)
    df_bnb['algorithm'] = 'B&B'
    print(f"Loaded B&B results: {len(df_bnb)} instances")
except FileNotFoundError:
    print(f"B&B results not found at {bnb_path}")
    df_bnb = pd.DataFrame()

try:
    df_dp = pd.read_csv(dp_path)
    df_dp['algorithm'] = 'DP'
    # Rename 'states' to 'nodes' for consistency
    if 'states' in df_dp.columns:
        df_dp['nodes'] = df_dp['states']
    print(f"Loaded DP results: {len(df_dp)} instances")
except FileNotFoundError:
    print(f"DP results not found at {dp_path}")
    df_dp = pd.DataFrame()

# Combine results
if not df_bnb.empty and not df_dp.empty:
    df_combined = pd.concat([df_bnb, df_dp], ignore_index=True)
    print(f"\nCombined: {len(df_combined)} total results")
elif not df_bnb.empty:
    df_combined = df_bnb
    print("Only B&B results available")
elif not df_dp.empty:
    df_combined = df_dp
    print("Only DP results available")
else:
    df_combined = pd.DataFrame()
    print("No results available")

df_combined

In [ ]:
if not df_combined.empty:
    print("="*60)
    print("PHASE 1 SUMMARY STATISTICS")
    print("="*60)
    
    for algo in df_combined['algorithm'].unique():
        df_algo = df_combined[df_combined['algorithm'] == algo]
        optimal_count = df_algo['optimal'].sum()
        total = len(df_algo)
        avg_time = df_algo[df_algo['time_s'] > 0]['time_s'].mean()
        
        print(f"\n{algo}:")
        print(f"  Instances solved: {total}")
        print(f"  Proven optimal: {optimal_count} ({optimal_count/total*100:.1f}%)")
        print(f"  Average time: {avg_time:.1f}s")
        print(f"  Timeouts: {total - optimal_count}")
else:
    print("No data to summarize")

In [ ]:
if not df_combined.empty and len(df_combined['algorithm'].unique()) > 1:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Optimality rate by algorithm
    optimal_by_algo = df_combined.groupby('algorithm')['optimal'].agg(['sum', 'count'])
    optimal_by_algo['rate'] = optimal_by_algo['sum'] / optimal_by_algo['count'] * 100
    
    ax1.bar(optimal_by_algo.index, optimal_by_algo['rate'], color=['#2ecc71', '#3498db'])
    ax1.set_ylabel('Optimality Proof Rate (%)')
    ax1.set_title('Optimality Proof Rate by Algorithm')
    ax1.set_ylim(0, 100)
    for i, v in enumerate(optimal_by_algo['rate']):
        ax1.text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')
    
    # Plot 2: Computation time comparison
    df_optimal = df_combined[df_combined['optimal'] == True]
    if not df_optimal.empty:
        df_optimal.boxplot(column='time_s', by='algorithm', ax=ax2)
        ax2.set_ylabel('Computation Time (s)')
        ax2.set_xlabel('Algorithm')
        ax2.set_title('Computation Time for Proven Optimal Solutions')
        plt.suptitle('')
    
    plt.tight_layout()
    plt.show()
else:
    print("Need results from both algorithms for comparison")

In [ ]:
if not df_combined.empty and len(df_combined['algorithm'].unique()) > 1:
    # Pivot to compare algorithms side-by-side
    df_pivot = df_combined.pivot_table(
        index='instance',
        columns='algorithm',
        values=['optimal', 'time_s'],
        aggfunc='first'
    )
    
    # Plot instances where both algorithms succeeded
    both_optimal = df_pivot[('optimal', 'B&B')] & df_pivot[('optimal', 'DP')]
    df_both = df_pivot[both_optimal]
    
    if not df_both.empty:
        fig, ax = plt.subplots(figsize=(12, 6))
        x = np.arange(len(df_both))
        width = 0.35
        
        ax.bar(x - width/2, df_both[('time_s', 'B&B')], width, label='B&B', color='#2ecc71')
        ax.bar(x + width/2, df_both[('time_s', 'DP')], width, label='DP', color='#3498db')
        
        ax.set_ylabel('Computation Time (s)')
        ax.set_title('Computation Time Comparison (Instances Proven Optimal by Both)')
        ax.set_xticks(x)
        ax.set_xticklabels(df_both.index, rotation=45, ha='right')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print(f"\nBoth algorithms proved optimality for {len(df_both)} instances")
    else:
        print("No instances where both algorithms proved optimality")
else:
    print("Need results from both algorithms for comparison")

In [ ]:
if not df_combined.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for algo in df_combined['algorithm'].unique():
        df_algo = df_combined[df_combined['algorithm'] == algo]
        df_algo_sorted = df_algo[df_algo['optimal'] == True].sort_values('time_s')
        
        if not df_algo_sorted.empty:
            cumulative = np.arange(1, len(df_algo_sorted) + 1)
            ax.plot(df_algo_sorted['time_s'], cumulative, marker='o', label=algo, linewidth=2)
    
    ax.set_xlabel('Computation Time (s)')
    ax.set_ylabel('Number of Instances Proven Optimal')
    ax.set_title('Performance Profile: Cumulative Optimality Proofs over Time')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xscale('log')
    
    plt.tight_layout()
    plt.show()
else:
    print("No data to plot")